# 환경 변수 세팅

In [1]:
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

True

In [2]:
import pandas as pd 

df = pd.read_csv("../static/SteamGames_cleaned.csv")
df.head(5)

,Appid,Name,Type,ReleaseDate,Developers,Publishers,Description,price,Thumbnail,ReviewScore,...,price_type,TotalReviews,ReviewRatio,ReviewConfidence,BayesianRating,PopularityScore,FreshnessScore,PriceBucket,AffordabilityScore,DiscoveryScore
0,3764200,Resident Evil Requiem,game,2026-02-26,"CAPCOM Co., Ltd.","CAPCOM Co., Ltd.",Resident Evil Requiem Deluxe Edition Resident ...,69.99,https://shared.akamai.steamstatic.com/store_it...,9,...,Paid,24508,0.967643,1.000000,0.965727,0.971084,1.000000,Premium,0.00000,0.875635
1,730,Counter-Strike 2,game,2012-08-21,Valve,Valve,"For over two decades, Counter-Strike has offer...",0.00,https://shared.akamai.steamstatic.com/store_it...,8,...,Free,1417290,0.857197,1.000000,0.857194,1.000000,0.143587,Free,1.00000,0.800135
2,1808500,ARC Raiders,game,2025-10-30,Embark Studios,Embark Studios,ARC Raiders Deluxe Edition ARC Raiders Deluxe ...,39.99,https://shared.akamai.steamstatic.com/store_it...,8,...,Paid,172725,0.865717,1.000000,0.865670,1.000000,0.870551,Premium,0.09667,0.823084
3,359550,Tom Clancy's Rainbow Six Siege,game,2015-12-01,Ubisoft Montreal,Ubisoft,Season 11 Year 1 Overview Year 11 Roadmap Free...,0.00,https://shared.akamai.steamstatic.com/store_it...,8,...,Free,629725,0.830340,1.000000,0.830349,1.000000,0.217638,Free,1.00000,0.797820
4,1172470,Apex Legends™,game,2020-11-04,Respawn,Electronic Arts,Apex Legends: Breach About the Game Conquer wi...,0.00,https://shared.akamai.steamstatic.com/store_it...,6,...,Free,1036,0.791506,0.931231,0.805892,0.630143,0.435275,Free,1.00000,0.725773


In [14]:
df['Type'].value_counts()

Type
game     22461
dlc       6539
music      585
mod         23
video        1
Name: count, dtype: int64

In [22]:
str(df['Appid'].min()).zfill(7)

'0000010'

# 주요 항목 

- Appid: 키값
- Name: 게임 이름 
- Type: game | dlc | music | mod | video
- Developers: 개발사
- Description: 설명 
- price: 가격
- Thumbnail: 썸내일
- tags: 게임 장르를 포함한 태그

## 노드 타입:

- Game: 게임(id, name, released, score, thumbnail), Type이 game인 경우에만 포함
- Company: 회사(name 속성)
- Genre: 장르 정보(name 속성) tags 항목 split 

## 관계 타입:

- DEVELOPED: Company -> Game
- PUBLISHED: Company -> Game 
- IN_GENRE: Game -> Genre

## 제약조건:

- (g:Game) g.id IS UNIQUE
- (c:Company) c.name IS UNIQUE
- (r:Genre) r.name IS UNIQUE 

## INDEX:
- game_name, game released
- company_name
- genre_name

# 인덱스 및 제약 조건 세팅

In [13]:
import os 

from langchain_neo4j import Neo4jGraph

# LangChain 도구 활용 - DB 연결 객체 초기화 
graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD"),
)

constraints = [
    "CREATE CONSTRAINT game_id_unique IF NOT EXISTS FOR (g:Game) REQUIRE g.id IS UNIQUE",
    "CREATE CONSTRAINT company_name_unique IF NOT EXISTS FOR (c:Company) REQUIRE c.name IS UNIQUE",
    "CREATE CONSTRAINT genre_name_unique IF NOT EXISTS FOR (g:Genre) REQUIRE g.name IS UNIQUE"
]

indexes = [
    "CREATE INDEX game_name_index IF NOT EXISTS FOR (g:Game) ON (g.name)",
    "CREATE INDEX game_release_index IF NOT EXISTS FOR (g:Game) ON (g.released)",
]

for constraint in constraints:
    graph.query(constraint)

for index in indexes:
    graph.query(index)

In [4]:
from langchain_neo4j.graphs.graph_document import GraphDocument, Node, Relationship

In [10]:
import re

node_dict = {}
relationships = []

total_rows = len(df)
batch_size = 100

for batch_start in range(0, total_rows, batch_size):
    batch_end = min(batch_start + batch_size, total_rows)
    batch_df = df.iloc[batch_start:batch_end]
    for _, row in batch_df.iterrows():
        if row['Type'] != 'game':
            continue
        app_id_str = str(row['Appid']).zfill(7)
        game_id = f"game-{app_id_str}"
        # Game 노드 추가
        if game_id not in node_dict:
            game_properties = {
                "id": game_id, 
                "name": row['Name'], 
                "released": row['ReleaseDate'], 
                "description": row['Description'],
                "thumbnail": row['Thumbnail']
            }
            
            if pd.notna(row.get('ReviewScore')):
                game_properties["score"] = float(row.get('ReviewScore'))
            
            game_node = Node(
                id=game_id,
                type="Game",
                properties=game_properties
            )
            node_dict[game_id] = game_node
        # Genre 노드 추가, IN_GENRE 관계 추가
        if pd.notna(row.get('tags')):
            for tag in row.get('tags').split(','):
                tag = tag.strip()
                tag_id = f"genre-{tag}"
                
                if tag_id not in node_dict:
                    genre_node = Node(
                        id=tag_id,
                        type="Genre",
                        properties={"name": tag}
                    )
                    node_dict[tag_id] = genre_node
                
                relationships.append(
                    Relationship(
                        source=node_dict[game_id],
                        target=node_dict[tag_id],
                        type="IN_GENRE",
                        properties={}
                    )
                )
        if pd.notna(row.get('Developers')):
            developer = row.get("Developers").strip()
            # developer = re.sub(r"[^\w\s가-힣]", "", developer).strip()
            company_id = f"company-{developer}"
            if company_id not in node_dict:
                company_node = Node(
                    id=company_id,
                    type="Company",
                    properties={"name": developer}
                )
                node_dict[company_id] = company_node
                
            relationships.append(
                Relationship(
                    source=node_dict[company_id],
                    target=node_dict[game_id],
                    type="DEVELOP",
                    properties={}
                )
            )
                
        if pd.notna(row.get('Publishers')):
            publisher = row.get("Publishers").strip()
            company_id = f"company-{publisher}"
            if company_id not in node_dict:
                company_node = Node(
                    id=company_id,
                    type="Company",
                    properties={"name": publisher}
                )
                node_dict[company_id] = company_node
                
            relationships.append(
                Relationship(
                    source=node_dict[company_id],
                    target=node_dict[game_id],
                    type="PUBLISH",
                    properties={}
                )
            )
    print(f"배치 처리 완료: {batch_start+1}~{batch_end}/{total_rows} 레코드")

# 결과 출력
print(f"총 노드 수: {len(node_dict)}")
print(f"총 관계 수: {len(relationships)}")

배치 처리 완료: 1~100/29609 레코드
배치 처리 완료: 101~200/29609 레코드
배치 처리 완료: 201~300/29609 레코드
배치 처리 완료: 301~400/29609 레코드
배치 처리 완료: 401~500/29609 레코드
배치 처리 완료: 501~600/29609 레코드
배치 처리 완료: 601~700/29609 레코드
배치 처리 완료: 701~800/29609 레코드
배치 처리 완료: 801~900/29609 레코드
배치 처리 완료: 901~1000/29609 레코드
배치 처리 완료: 1001~1100/29609 레코드
배치 처리 완료: 1101~1200/29609 레코드
배치 처리 완료: 1201~1300/29609 레코드
배치 처리 완료: 1301~1400/29609 레코드
배치 처리 완료: 1401~1500/29609 레코드
배치 처리 완료: 1501~1600/29609 레코드
배치 처리 완료: 1601~1700/29609 레코드
배치 처리 완료: 1701~1800/29609 레코드
배치 처리 완료: 1801~1900/29609 레코드
배치 처리 완료: 1901~2000/29609 레코드
배치 처리 완료: 2001~2100/29609 레코드
배치 처리 완료: 2101~2200/29609 레코드
배치 처리 완료: 2201~2300/29609 레코드
배치 처리 완료: 2301~2400/29609 레코드
배치 처리 완료: 2401~2500/29609 레코드
배치 처리 완료: 2501~2600/29609 레코드
배치 처리 완료: 2601~2700/29609 레코드
배치 처리 완료: 2701~2800/29609 레코드
배치 처리 완료: 2801~2900/29609 레코드
배치 처리 완료: 2901~3000/29609 레코드
배치 처리 완료: 3001~3100/29609 레코드
배치 처리 완료: 3101~3200/29609 레코드
배치 처리 완료: 3201~3300/29609 레코드
배치 처리 완료: 3301~3400/29609 레코드
배

In [15]:
nodes = list(node_dict.values())

graph_doc = GraphDocument(
    nodes=nodes,
    relationships=relationships
)

graph.add_graph_documents([graph_doc])
print("그래프 데이터베이스에 저장 완료")

그래프 데이터베이스에 저장 완료
